# Karaoke Text Processing

In [84]:
## Credit 
## https://github.com/nomadkaraoke/karaoke-lyrics-processor

import re
import logging
import os
import codecs

class KaraokeLyricsProcessor:
    def __init__(
        self,
        log_level=logging.INFO,
        log_formatter=None,
        input_lyrics_text=None,
        max_line_length=36,
        max_line_length_cjk=12
    ):
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(log_level)
        self.logger.propagate = False
        self.log_level = log_level
        self.log_formatter = log_formatter

        self.log_handler = logging.StreamHandler()

        if self.log_formatter is None:
            self.log_formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(module)s - %(message)s")

        self.log_handler.setFormatter(self.log_formatter)
        # Re-running the notebook cell reuses the same module logger, so
        # normalize it back to one processor-owned handler.
        existing_handler = next(
            (handler for handler in self.logger.handlers if getattr(handler, "_karaoke_processor_handler", False)),
            None,
        )
        if existing_handler is None:
            for handler in list(self.logger.handlers):
                self.logger.removeHandler(handler)
            self.log_handler._karaoke_processor_handler = True
            self.logger.addHandler(self.log_handler)
        else:
            self.log_handler = existing_handler

        self.logger.debug(f"Karaoke Lyrics Processor instantiating with max_line_length: {max_line_length}")

        self.max_line_length = max_line_length
        self.max_line_length_cjk = max_line_length_cjk

        self.input_lyrics_text = input_lyrics_text

        self.input_lyrics_lines = self.input_lyrics_text.splitlines() if self.input_lyrics_text else []
        self.cjk_regex = re.compile(r'[\u4e00-\u9fff]+')



    def clean_text(self, text):
        # Remove any non-printable characters except newlines and U+2005
        original_len = len(text)
        cleaned = "".join(char for char in text if char.isprintable() or char in ["\n", "\u2005"])
        if len(cleaned) != original_len:
            self.logger.debug(f"Removed {original_len - len(cleaned)} non-printable characters")

        # Replace multiple newlines with a single newline
        newlines_before = cleaned.count("\n")
        cleaned = re.sub(r"\n{2,}", "\n", cleaned)
        newlines_after = cleaned.count("\n")
        if newlines_before != newlines_after:
            self.logger.debug(f"Consolidated {newlines_before - newlines_after} extra newlines")

        # Remove leading/trailing whitespace from each line
        lines_before = cleaned.splitlines()
        cleaned = "\n".join(line.strip() for line in lines_before)
        lines_after = cleaned.splitlines()

        # Count lines that changed due to stripping
        changed_lines = sum(1 for before, after in zip(lines_before, lines_after) if before != after)
        if changed_lines > 0:
            self.logger.debug(f"Stripped whitespace from {changed_lines} lines")

        return cleaned

    def find_best_split_point(self, line):
        """
        Find the best split point in a line based on the specified criteria.
        """

        max_line_length = self.max_line_length_cjk if self.cjk_regex.search(line) else self.max_line_length

        self.logger.debug(f"Finding best_split_point for line: {line}")
        words = line.split()
        mid_word_index = len(words) // 2
        self.logger.debug(f"words: {words} mid_word_index: {mid_word_index}")

        # Check for a comma within one or two words of the middle word
        if "," in line:
            mid_point = len(" ".join(words[:mid_word_index]))
            comma_indices = [i for i, char in enumerate(line) if char == ","]

            for index in comma_indices:
                if abs(mid_point - index) < 20 and len(line[: index + 1].strip()) <= max_line_length:
                    self.logger.debug(
                        f"Found comma at index {index} which is within 20 characters of mid_point {mid_point} and results in a suitable line length, accepting as split point"
                    )
                    return index + 1  # Include the comma in the first line

        # Check for 'and'
        if " and " in line:
            mid_point = len(line) // 2
            and_indices = [m.start() for m in re.finditer(" and ", line)]
            for index in sorted(and_indices, key=lambda x: abs(x - mid_point)):
                if len(line[: index + len(" and ")].strip()) <= max_line_length:
                    self.logger.debug(f"Found 'and' at index {index} which results in a suitable line length, accepting as split point")
                    return index + len(" and ")

        # If no better split point is found, try splitting at the middle word
        if len(words) > 2 and mid_word_index > 0:
            split_at_middle = len(" ".join(words[:mid_word_index]))
            if split_at_middle <= max_line_length:
                self.logger.debug(f"Splitting at middle word index: {mid_word_index}")
                return split_at_middle

        # If the line is still too long, find the last space before max_line_length
        if len(line) > max_line_length:
            last_space = line.rfind(" ", 0, max_line_length)
            if last_space != -1:
                self.logger.debug(f"Splitting at last space before max_line_length: {last_space}")
                return last_space
            else:
                # If no space is found, split at max_line_length
                self.logger.debug(f"No space found, forcibly splitting at max_line_length: {max_line_length}")
                return max_line_length

        # If the line is shorter than max_line_length, return its length
        return len(line)

    def replace_non_printable_spaces(self, text):
        """
        Replace non-printable space-like characters, tabs, and other whitespace with regular spaces,
        excluding newline characters.
        """
        # Log each character and its Unicode code point
        # for i, char in enumerate(text):
        #     self.logger.debug(f"Character at position {i}: {repr(char)} (Unicode: U+{ord(char):04X})")

        # Define a pattern for space-like characters, including tabs and other whitespace, but excluding newlines
        space_pattern = r"[^\S\n]|\u00A0|\u1680|\u2000-\u200A|\u202F|\u205F|\u3000"

        # Replace matched characters with a regular space
        cleaned_text = re.sub(space_pattern, " ", text)

        # Remove leading/trailing spaces and collapse multiple spaces into one, preserving newlines
        final_text = re.sub(r" +", " ", cleaned_text).strip()

        return final_text

    def clean_punctuation_spacing(self, text):
        """
        Remove unnecessary spaces before punctuation marks.
        """
        self.logger.debug(f"Cleaning punctuation spacing")
        # Remove space before comma, period, exclamation mark, question mark, colon, and semicolon
        cleaned_text = re.sub(r"\s+([,\.!?:;])", r"\1", text)

        return cleaned_text

    def fix_commas_inside_quotes(self, text):
        """
        Move commas inside quotes to after the closing quote.
        """
        self.logger.debug(f"Fixing commas inside quotes")
        # Use regex to find patterns where a comma is inside quotes and move it outside
        fixed_text = re.sub(r'(".*?)(,)(\s*")', r"\1\3\2", text)

        return fixed_text

    def process_line(self, line):
        """
        Process a single line to ensure it's within the maximum length,
        handle parentheses, and replace non-printable spaces.
        """

        max_line_length = self.max_line_length_cjk if self.cjk_regex.search(line) else self.max_line_length

        line = self.replace_non_printable_spaces(line)
        line = self.clean_punctuation_spacing(line)
        line = self.fix_commas_inside_quotes(line)

        processed_lines = []
        iteration_count = 0
        max_iterations = 100  # Failsafe limit

        while self.line_too_long(line) and iteration_count < max_iterations:
            # Check if the line contains parentheses
            if "(" in line and ")" in line:
                start_paren = line.find("(")
                end_paren = self.find_matching_paren(line, start_paren)
                if end_paren < len(line) and line[end_paren] == ",":
                    end_paren += 1

                # Process text before parentheses if it exists
                if start_paren > 0:
                    before_paren = line[:start_paren].strip()
                    processed_lines.extend(self.split_line(before_paren))

                # Process text within parentheses
                paren_content = line[start_paren : end_paren + 1].strip()
                if self.line_too_long(paren_content):
                    # Split the content within parentheses if it's too long
                    split_paren_content = self.split_line(paren_content)
                    processed_lines.extend(split_paren_content)
                else:
                    processed_lines.append(paren_content)

                line = line[end_paren + 1 :].strip()
            else:
                split_point = self.find_best_split_point(line)
                # Ensure we make progress - if split_point is 0 or too small, force a reasonable split
                if split_point <= 0:
                    split_point = min(max_line_length, len(line))
                processed_lines.append(line[:split_point].strip())
                line = line[split_point:].strip()

            iteration_count += 1

        if line:  # Add any remaining part
            processed_lines.extend(self.split_line(line))

        if iteration_count >= max_iterations:
            self.logger.error(f"Maximum iterations exceeded in process_line for line: {line}")

        return processed_lines

    def find_matching_paren(self, line, start_index):
        """
        Find the index of the matching closing parenthesis for the opening parenthesis at start_index.
        """
        stack = 0
        for i in range(start_index, len(line)):
            if line[i] == "(":
                stack += 1
            elif line[i] == ")":
                stack -= 1
                if stack == 0:
                    return i
        return -1  # No matching parenthesis found

    def split_line(self, line) -> list:
        """
        Split a line into multiple lines if it exceeds the maximum length.
        """

        max_line_length = self.max_line_length_cjk if self.cjk_regex.search(line) else self.max_line_length

        if not self.line_too_long(line):
            return [line]

        split_lines = []
        while self.line_too_long(line):
            split_point = self.find_best_split_point(line)
            # Ensure we make progress - if split_point is 0 or too small, force a reasonable split
            if split_point <= 0:
                split_point = min(max_line_length, len(line))
            split_lines.append(line[:split_point].strip())
            line = line[split_point:].strip()

        if line:
            split_lines.append(line)

        return split_lines
    
    def line_too_long(self, line):
        """
        Check if a line exceeds the maximum length, considering CJK characters.
        """
        if self.cjk_regex.search(line):
            return len(line) > self.max_line_length_cjk
        else:
            return len(line) > self.max_line_length

    def process(self):

        lyrics_lines = self.input_lyrics_lines
        processed_lyrics_text = ""
        iteration_count = 0
        max_iterations = 100  # Failsafe limit

        all_processed = False
        while not all_processed:
            if iteration_count > max_iterations:
                self.logger.error("Maximum iterations exceeded while processing lyrics.")
                break

            all_processed = True
            new_lyrics = []
            previous_line_count = len(lyrics_lines)

            for line in lyrics_lines:
                line = line.strip()
                processed = self.process_line(line)
                new_lyrics.extend(processed)
                if any(self.line_too_long(l) for l in processed):
                    print(f"Line still too long after processing: {processed}")
                    all_processed = False

            lyrics_lines = new_lyrics

            # Safety check: if no progress is being made, break out of the loop
            if len(lyrics_lines) == previous_line_count and not all_processed:
                self.logger.warning("No progress made in processing, forcing completion to avoid infinite loop")
                break

            iteration_count += 1

        processed_lyrics_lines = [self.replace_non_printable_spaces(line) for line in lyrics_lines]
        processed_lyrics_lines = [self.clean_punctuation_spacing(line) for line in processed_lyrics_lines]
    
        return processed_lyrics_lines

        processed_lyrics_text = "\n".join(lyrics_lines)

        # Final pass to replace any remaining non-printable spaces and clean punctuation
        processed_lyrics_text = self.replace_non_printable_spaces(processed_lyrics_text)
        processed_lyrics_text = self.clean_punctuation_spacing(processed_lyrics_text)

        self.processed_lyrics_text = processed_lyrics_text

        return processed_lyrics_text



### Processing plain text
Input is just a multiline string, not WhisperX segments.

Returns: list of lines

In [74]:
# Not the full lyrics, just a sample with some edge cases
input_lyrics_text="""
We were good, we were gold
Kinda dream that can't be sold
We were right 'til we weren't
Built a home and watched it burn
Mm, I didn't wanna leave you, I didn't wanna lie
Started to cry, but then remembered I
I can buy myself flowers
Write my name in the sand
Talk to myself for hours
Say things you don't understand
I can take myself dancing
Can love me better, I can love me better, baby, Can love me better, I can love me better, baby, Can love me better, I can love me better, baby 
Can love me better, I can love me better, baby
Can love me better, I can love me better, baby
Can love me better, I- (ooh, I)
I didn't wanna leave you, I didn't wanna fight
Yeah, I can love me better than you can
Can love me better, I can love me better, baby (oh, oh)
Can love me better, I can love me better, baby (than you can)
Can love me better, I can love me better, baby
Can love me better, I-
"""

In [75]:
processor = KaraokeLyricsProcessor(max_line_length=36, input_lyrics_text=input_lyrics_text)
processed_lines = processor.process()
processed_lines

['We were good, we were gold',
 "Kinda dream that can't be sold",
 "We were right 'til we weren't",
 'Built a home and watched it burn',
 "Mm, I didn't wanna leave you,",
 "I didn't wanna lie",
 'Started to cry,',
 'but then remembered I',
 'I can buy myself flowers',
 'Write my name in the sand',
 'Talk to myself for hours',
 "Say things you don't understand",
 'I can take myself dancing',
 'Can love me better, I can love me',
 'better, baby, Can love me better, I',
 'can love me better,',
 'baby, Can love me better,',
 'I can love me better, baby',
 'Can love me better,',
 'I can love me better, baby',
 'Can love me better,',
 'I can love me better, baby',
 'Can love me better, I- (ooh, I)',
 "I didn't wanna leave you,",
 "I didn't wanna fight",
 'Yeah,',
 'I can love me better than you can',
 'Can love me better,',
 'I can love me better, baby',
 '(oh, oh)',
 'Can love me better,',
 'I can love me better, baby',
 '(than you can)',
 'Can love me better,',
 'I can love me better, baby

For CJK (Chinese, Japanese, Korean), the argument is explicitly set to `max_line_length_cjk` instead of `max_line_length`. The default value is 12, which is the maximum number of characters per line for CJK. For non-CJK, the default value is 36.

In [82]:
# Chinese example
chinese_input = """
風到這裡就是黏 黏住過客的思念
雨到了這裡纏成線 纏著我們流連人世間
你在身邊就是緣 緣份寫在三生石上面
愛有萬分之一甜 寧願我就葬在這一點
圈圈圓圓圈圈 天天年年天天
的我 深深看你的臉
生氣的溫柔 埋怨的溫柔 的臉
離愁能有多痛 痛有多濃
當夢被埋在江南煙雨中 心碎了才懂
♪
相信愛一天 抵過永遠
在這一剎那凍結了時間
不懂怎麼表現溫柔的我們
還以為殉情只是古老的傳言
離愁能有多痛 痛有多濃
當夢被埋在江南煙雨中 心碎了才懂
"""

In [83]:
processor = KaraokeLyricsProcessor(max_line_length_cjk=13, input_lyrics_text=chinese_input)
processor.process()

['風到這裡就是黏',
 '黏住過客的思念',
 '雨到了這裡纏成線',
 '纏著我們流連人世間',
 '你在身邊就是緣',
 '緣份寫在三生石上面',
 '愛有萬分之一甜',
 '寧願我就葬在這一點',
 '圈圈圓圓圈圈 天天年年天天',
 '的我 深深看你的臉',
 '生氣的溫柔',
 '埋怨的溫柔 的臉',
 '離愁能有多痛 痛有多濃',
 '當夢被埋在江南煙雨中',
 '心碎了才懂',
 '♪',
 '相信愛一天 抵過永遠',
 '在這一剎那凍結了時間',
 '不懂怎麼表現溫柔的我們',
 '還以為殉情只是古老的傳言',
 '離愁能有多痛 痛有多濃',
 '當夢被埋在江南煙雨中',
 '心碎了才懂']

These code below is using for testing, it tests a range of max_line_length(s) and grabs the basic statistics of each.

This may be useful for future implementation, where the user specify a maximum number of lines, and the program will try different values around that to find the best fit. That is
- provide the most consistent line length
- reduce edge cases where a line is too long or too short

In [ ]:
import statistics

for i in range(25,48):
    processor = KaraokeLyricsProcessor(max_line_length=i, input_lyrics_text=input_lyrics_text)
    processed_text = processor.process()
    print(processed_text)
    word_length = [len(line.split()) for line in processed_text]
    min_length = min(word_length)
    print(processed_text[word_length.index(min_length)])
    print(f"Max word length: {i}, Min: {min(word_length)}, Avg: {statistics.mean(word_length)}")

## Processing WhisperX Input
This section outlines the steps to process WhisperX input and returns the rebuilt lyrics text.

```json
  {
    "start": 0.76,
    "end": 0.88,
    "text": "KSR",
    "words": [
      {
        "word": "KSR",
        "start": 0.76,
        "end": 0.88,
        "score": 0.36
      }
    ]
  }
```

In [23]:
import json
with open("example.json","r", encoding="utf-8") as f:
    example = json.load(f)

In [ ]:
from dataclasses import dataclass

# This Segment class is used (same as demucs_svc) for compatibility with realign functions due to the function input, without rewriting the function.
@dataclass
class Segment:
    text: str | None
    start: float
    end: float

Uses WhisperX segments to build the plaintext lyrics while using the nested individual word as a list.

In [ ]:
# Original WhisperX input, flattened to all the words without segments
words = [word for segment in example for word in segment["words"]]
words

In [ ]:
# Use the original WhisperX segments to build a multi-line lyrics text
segments = [s['text'] for s in example]
lyrics_text = "\n".join(segments)

In [86]:
# Since the lyrics text is already a multi-line string, same processing pipeline as above
processor = KaraokeLyricsProcessor(max_line_length=36, max_line_length_cjk=12, input_lyrics_text=lyrics_text)
processed_lines = processor.process()
processed_lines

['KSR',
 "It's Cardi, ayy",
 'Said, "I\'m the shit,',
 "they can't fuck with",
 'me if they wanted to"',
 'God damn',
 'Said, "Lil bitch,',
 "you can't fuck with",
 'me if you wanted to"',
 'These expensive,',
 'these is red bottoms,',
 'these is bloody shoes',
 'Hit the store,',
 "I can get 'em both,",
 "I don't wanna choose",
 "And I'm quick,",
 'cut a nigga off,',
 "so don't get comfortable, look",
 "I don't dance now,",
 'I make money moves',
 "Say I don't gotta dance,",
 'I make money move',
 "If I see you and I don't speak,",
 "that means I don't fuck with you",
 "I'm a boss,",
 'you a worker,',
 'bitch, I make bloody moves',
 "Now she say she gon' do what to who?",
 "Let's find out and see",
 "Cardi B, you know where I'm at,",
 'you know where I be',
 'You in the club just to party,',
 "I'm there, I get paid a fee",
 'I be in and out them banks so much',
 "I know they're tired of me",
 'Honestly,',
 "don't give a fuck 'bout",
 "who ain't fond of me",
 'Dropped two mixtapes in si

In [ ]:
# Compare the number of words in the original WhisperX input with the number of words in the processed output
sub_segments = [l for l in processed_lines if l and l.strip()]
from itertools import chain
len(words), len(list(chain.from_iterable([l.split() for l in sub_segments])))

(689, 689)

In [89]:
# recreate the segments as a dataclass for compatibility with realign functions
segments = [Segment(text=segment,start=0,end=0) for segment in sub_segments]

In [ ]:
# rebuild the segments with the new line-level alignment (this is the same code from demucs_svc)

from collections.abc import Iterable
from typing import Any

def join_display_tokens(tokens: Iterable[str]) -> str:
    tokens = list(tokens)
    if any('\u4e00' <= ch <= '\u9fff' for token in tokens for ch in token):
        return ''.join(tokens)
    return ' '.join(tokens)

def make_segment(
        start: int, # start index
        end: int,  # end index
        words: list[dict[str, Any]] # list of word-level alignment results, from whisper output 
    ) -> dict[str, Any]:
    """
    [{start,end,text},...] -> [{start,end,text,words:[{word,start,end},...]}]
    """
    curr_words = [w for w in words[start:end] if w["word"].strip()]
    words_start = words[start]["start"] if words else 0.0
    words_end = words[end-1]["end"] if words else 0.0
    return {
        "start": words_start,
        "end": words_end,
        "text": join_display_tokens(w["word"] for w in curr_words),
        "words": curr_words
    }

In [51]:
TOKEN_RE = re.compile(
    r"""
    [\u4e00-\u9fff]                       # Chinese Han char
    | [A-Za-z]+(?:['’][A-Za-z]+)*         # English words with contractions
    | \d+(?:[.,]\d+)*                     # numbers
    | [^\s]                               # fallback punctuation/symbol
    """,
    re.VERBOSE,
)

def lyric_tokens(text: str) -> list[str]:
    return TOKEN_RE.findall(text)

def alignment_tokens(text: str) -> list[str]:
    return [
        token for token in lyric_tokens(text)
        if re.search(r"[\u4e00-\u9fffA-Za-z0-9]", token)
    ]

def realign_easy(segments: list[Segment], words: list[dict]) -> list[dict[str, Any]]:
    """
    The length of words and segments match, so we can just assign words to each segment based on the word count of each segment
    """
    new_segments = []
    curr_idx = 0
    for segment in segments:
        segment_word_count = len(alignment_tokens(segment.text)) # type: ignore
        prev_idx = curr_idx
        curr_idx += segment_word_count
        print(f"Line: {segment.text}, Start: {prev_idx}, End: {curr_idx}")
        print(f"Aligned words for this line: {[w['word'] for w in words[prev_idx:curr_idx]]}")
        new_segments.append(make_segment(prev_idx, curr_idx, words))
    return new_segments 

def realign_hard(segments: list[Segment], words: list[dict]) -> list[dict[str, Any]]:
    """
    If the length of words in the original segments don't match the length of words in the whisper output
    """
    def clean_token(s: str) -> str:
        return re.sub(r"[^\w']+", "", s).lower()

    idxs = []
    new_segments = []
    w_idx = 0
    total_words = len(words)

    for lrc_line in segments:
        tokens = [t for t in (clean_token(t) for t in alignment_tokens(lrc_line.text)) if t]
        line_start = w_idx
        for tok in tokens:
            # advance until we find a matching cleaned token or run out of words
            while w_idx < total_words and clean_token(words[w_idx]["word"]) != tok:
                w_idx += 1
            if w_idx >= total_words:
                break
            # matched this token, consume the word
            w_idx += 1
        idxs.append((line_start, w_idx))
    for (start, end) in idxs:
        print(f"Line: {segments[idxs.index((start,end))].text}, Start: {start}, End: {end}")
        print(f"Aligned words for this line: {[w['word'] for w in words[start:end]]}")
        new_segments.append(make_segment(start, end, words))
    return new_segments

In [ ]:
# same logic as demucs_svc, if length same we can lazily realign and join, otherwise
if len(words) == len(list(chain.from_iterable([l.split() for l in sub_segments]))):
    new_segments = realign_easy(segments, words)
else:
    new_segments = realign_hard(segments, words)

new_segments

In [72]:
with open("processed.json", "w", encoding="utf-8") as f:
    json.dump(new_segments, f, ensure_ascii=False, indent=4)

## Other Helpful

Refer to
- `text_processing.js` for logic of merge and split
- `diff.js` for logic of diff and patch (buggy, for initial testing only), mostly around DOM manipulation too